# 4 — Full Pipeline Integration

Tests the integrated system with both components (Bingham filter + T³ QAN) on the 2D baseline arena (not running the experiment).

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../.."))

from network.visualize3D import visualize_trajectory_projections, plot_pi_error
from experiments.arena_2d import Arena2DExperiment, Arena2DConfig
from analysis.scoring import score_2d
from metrics import wrapped_angle_diff
from config import RunConfig, NetworkConfig, ExperimentConfig
from experiments.arena_2d import Arena2DExperiment




In [ ]:
# config + experiment
g_vec = np.array([0., 0., -9.81])
cfg = RunConfig(
    network=NetworkConfig(),                 # validated defaults
    experiment=Arena2DConfig(n_steps=5000), # Arena2DConfig → has run_name
)
exp = Arena2DExperiment(cfg, record=True)    # config, not g


## Set up the arena

In [ ]:
scale = cfg.experiment.scale
print("scale:", scale, " speed:", cfg.experiment.speed,
      " target_speed_rad:", getattr(cfg, "target_speed_rad", None))
print("commanded theta_dot = scale*speed =", scale * cfg.experiment.speed, "rad/step")
print("ceiling = 0.0168  -> saturated?" , scale*cfg.experiment.speed > 0.0168)

In [ ]:
# run + save
print("Running ...")
t0 = time.time()
result = exp.run_experiment(g=g_vec)
exp.save(result, f"runs/{cfg.experiment.run_name}.npz")
print(f"Saved runs/{cfg.experiment.run_name}.npz  ({time.time()-t0:.1f}s)")

# everything from the result object (single source of truth)
world_pos  = result.world_pos
torus_gt   = result.torus_gt
theta_hist = result.theta_hist
gap        = result.gap_hist
n_hat_hist = result.n_hat_hist
T          = cfg.experiment.n_steps

print(f"Filter gap  t=100: {gap[99]:.4f}  t=500: {gap[499]:.4f}  t={T}: {gap[-1]:.4f}  target > 2 by t=500")
print(f"n_hat at t={T}: {n_hat_hist[-1]}  (target [0, 0, 1])")
print(f"theta_3 decoded range: [{theta_hist[:, 2].min():.4f}, {theta_hist[:, 2].max():.4f}]  target ~[0, 0]")
print(f"MADE error  full: {result.mean_norm_error:.4f}  after t=200: {result.norm_error[200:].mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(world_pos[:, 0], world_pos[:, 1], lw=0.5, alpha=0.7, color="steelblue")
ax.scatter(*world_pos[0, :2],  color="green", s=60, zorder=5, label="start")
ax.scatter(*world_pos[-1, :2], color="red",   s=60, zorder=5, label="end")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Physical world trajectory (flat 2D arena)")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, _ = visualize_trajectory_projections(
    torus_gt, theta_hist,
    title="2D baseline arena, ground truth vs decoded on the torus manifold",
)
plt.show()


In [ ]:
fig, _ = plot_pi_error(torus_gt, theta_hist)
plt.show()

In [ ]:
data = np.load(f"runs/{cfg.experiment.run_name}.npz")
S = data["S_tot_buffer"]           # (T, N)


In [ ]:
out = score_2d(f"runs/{cfg.experiment.run_name}.npz", norm="minmax")
f, ac = out["f"], out["ac"]
print(f"{f.shape[-1]} cells | occupancy {out['occupancy']:.1%}")

In [ ]:
print("scale:", cfg.scale, " speed:", cfg.speed,
      " target_speed_rad:", getattr(cfg, "target_speed_rad", None))
print("commanded theta_dot = scale*speed =", cfg.scale * cfg.speed, "rad/step")
print("ceiling = 0.0168  -> saturated?" , cfg.scale*cfg.speed > 0.0168)